# 1. System Description
In this example, we will work with analysing a magnetic levitation system (in 2D), as illustrated in the figure.


TODO: Add a video here of the REAL system.

The system consists of one levitating disk magnet that is free to rotate and move in space given the force it feels from two permanent/electromagnets placed below it. Our goal is to use the sensor measurements of the magnetic field in the origin to control the current in the electromagnets so that the levitating magnet actually levitates.

Given a state vector  

$$
\zeta = [x, z, \theta, \dot{x}, \dot{z}, \dot{\theta}]^\top,
$$

where:
- $(x,z)$ is the **position** of the levitating magnet,
- $(\dot{x},\dot{z})$ is the **velocity** of the levitating magnet,
- $\theta$ is the **angle** of the magnet w.r.t. the plane (CW),
- $\dot{\theta}$ is the **angular velocity** of the magnet,  

this system can be described by the following ordinary differential equations:

$$
\begin{aligned}
\begin{bmatrix}
\ddot{x}\\
\ddot{z}
\end{bmatrix}
&= \frac{1}{M}\sum_{i=1}^{n_u} \mathbf{F}_{i} + \mathbf{g},\\
%
\ddot{\theta} &= \frac{1}{J}\sum_{i=1}^{n_u}\tau_{i},
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbf{F}_{i} &= \frac{3 \mu_0}{4 \pi d_i^5}\Biggl[\left(\mathbf{m}_i \cdot \mathbf{d}_i\right) \mathbf{m}+\left(\mathbf{m} \cdot \mathbf{d}_i\right) \mathbf{m}_i+\left(\mathbf{m}_i \cdot \mathbf{m}\right) \mathbf{d}_i-\frac{5\left(\mathbf{m}_i \cdot \mathbf{d}_i\right)\left(\mathbf{m} \cdot \mathbf{d}_i\right)}{d_i^2} \mathbf{d}_i\Biggr],\\
%
\tau_{i} &= \frac{\mu_0}{4 \pi d_i^3} \left[\mathbf{m} \times\left(\frac{3\left(\mathbf{m}_i \cdot \mathbf{d}_i\right)\mathbf{d}_i}{d_i^2}-\mathbf{m}_i\right)\right].
\end{aligned}
$$

Here, we define the 2D cross product as
$$
\mathbf{a} \times \mathbf{b} := a_1 b_2 - a_2 b_1,
$$

and

$$
\mathbf{m} := m\begin{bmatrix}-\sin \theta \\ \cos \theta\end{bmatrix}, \quad \mathbf{m}_i := \begin{bmatrix}0 \\ m_i + kI_i\end{bmatrix},
$$

$$
\mathbf{d}_i := \mathbf{r} - \mathbf{r}_i, \quad d_i = \|\mathbf{d}_i\|.\quad \quad \quad
$$

Here:
- $m$ is the **magnitude of the magnetic moment** of the levitating magnet,
- $m_i$ is the **magnitude of the magnetic moment** of the supporting magnets,
- $M$ is the **mass** of the levitating magnet,
- $J$ is the **inertia** of the levitating magnet,
- $\mathbf{r}$ is the $(x,z)$ position of the levitating magnet,
- $\mathbf{r}_i$ is the $(x,z)$ position of the supporting magnets
- $g$ is the **gravitational acceleration**,
- $\mu_0$ is the **magnetic permeability of free space**
- $\mathbf{F}_i$ and $\mathbf{\tau}$ are the **force and torque** from the i'th magnet acting on the levitating magnet,
- $I_i$ is the **current** in the i'th solenoid/magnet
- k is some gemoetrical constant determining relation between current and the magnetic field produced by the solenoids in the model.

A magnetic sensor is also placed in the center of the based of the system, and its measurements can be modeled as

$$
    \mathbf{B}:= \frac{\mu_0}{4\pi r^3}\left[3\frac{(\mathbf{m}\cdot\mathbf{r})\mathbf{r}}{r^2}-\mathbf{m}\right].
$$

In the following, we will use this model to analyze the magnetic leviation system and design controllers for it.

In [ ]:
%pip install ipyquizjb
from ipyquizjb.questions import display_questions, display_package

display_package({
    "questions": [
        {
            "type": "MULTIPLE_CHOICE",
            "body": "What type of model is this?",
            "when" : "initial",
            "answers" : [
                "linear autonomous",
                "linear non-autonomous",
                "nonlinear autonomous",
                "nonlinear non-autonomous"
            ],
            "answer" : ["nonlinear non-autonomous"]
        },
        {
            "type" : "MULTIPLE_CHOICE",
            "body" : "Is this a reasonable physical model of the system?",
            "when" : "initial",
            "answers" : [
                "Yes",
                "No"
            ],
            "answer" : ["No"]
        },
        {
            "type" : "MULTIPLE_CHOICE",
            "body" : "What is the primary role of the magnetic sensor placed at the center of the system's base?",
            "when" : "retry",
            "answers" : [
                "To directly measure the position $(x,z)$ of the levitating magnet",
                "To measure the magnetic field $\\mathbf{B}$ for state estimation and control",
                "To provide a counteracting magnetic force to stabilize the levitating magnet",
                "To measure the angular velocity $\\dot{\\theta}$ of the levitating magnet",
                "I do not know"
            ],
            "answer" : ["To measure the magnetic field $\\mathbf{B}$ for state estimation and control"],
            "notes" : ["The magnetic sensor measures the magnetic field $\\mathbf{B}$ at its location, which can then be used (along with the model) to estimate the state of the levitating magnet for control purposes. It does not directly measure position or angular velocity, nor does it provide any counteracting force."]
        },
        {
            "type" : "MULTIPLE_CHOICE",
            "body" : "In the force equation $\mathbf{F}_i$, what happens to the force magnitude as the distance $d_i$ between magnets increases?",
            "when" : "retry",
            "answers" : [
                "The force increases linearly with distance",
                "The force remains constant regardless of distance",
                "The force decreases with the fourth power of distance ($\\sim 1/d_i^4$)",
                "The force decreases exponentially with distance",
                "I do not know"
            ],
            "answer" : ["The force decreases with the fourth power of distance ($\\sim 1/d_i^4$)"],
            "notes" : "The force $\mathbf{F}_i$ has terms with $d_i^5$ in the denominator, but the numerator contains $\\mathbf{d}_i$ (which is $\\sim d_i$), making the overall dependence $\\sim 1/d_i^4$. This inverse quartic relationship means the force decreases rapidly with distance."
        },
        {
            "type" : "MULTIPLE_CHOICE",
            "body" : "In the force equation $\mathbf{F}_i$, what happens to the force magnitude as the distance $d_i$ between magnets increases?",
            "when" : "retry",
            "answers" : [
                "The force increases linearly with distance",
                "The force remains constant regardless of distance",
                "The force decreases with the fourth power of distance ($\\sim 1/d_i^4$)",
                "The force decreases exponentially with distance",
                "I do not know"
            ],
            "answer" : ["The force decreases with the fourth power of distance ($\\sim 1/d_i^4$)"],
            "notes" : "The force $\mathbf{F}_i$ has terms with $d_i^5$ in the denominator, but the numerator contains $\\mathbf{d}_i$ (which is $\\sim d_i$), making the overall dependence $\\sim 1/d_i^4$. This inverse quartic relationship means the force decreases rapidly with distance."
        },
    ]
})